In [1]:
import os
from ingestion import load_data
from embeddings import create_vector_store
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings

In [2]:
VECTORSTORE_PATH = "vectorstore"
EMBED_MODEL      = "nomic-embed-text"

In [3]:
embeddings = OllamaEmbeddings(model=EMBED_MODEL)

if os.path.exists(VECTORSTORE_PATH):
    print("Loading cached vectorstore from disk...")
    vectorstore = FAISS.load_local(
        VECTORSTORE_PATH,
        embeddings,                        # <-- was missing, caused crash
        allow_dangerous_deserialization=True
    )
    print("Vectorstore loaded.")
else:
    print("No cached vectorstore found. Building from scratch (runs once)...")
    df = load_data()
    vectorstore = create_vector_store(df)
    vectorstore.save_local(VECTORSTORE_PATH)
    print(f"Vectorstore saved to '{VECTORSTORE_PATH}'.")


No cached vectorstore found. Building from scratch (runs once)...
Found 6 file(s).
Raw rows across all files: 110,626
Clean rows after filtering & dedup: 101,015


Building documents: 100%|██████████| 15771/15771 [00:11<00:00, 1390.57it/s]


Built 15,771 documents from 101,015 play events.
Embedding 15,771 documents in batches of 64...


Embedding batches: 100%|██████████| 246/246 [03:01<00:00,  1.36it/s]

Embedding complete.
Vectorstore saved to 'vectorstore'.


In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 6})
query = "How has my music taste evolved over time?"
docs  = retriever.invoke(query)

print(f"\nTop results for: '{query}'\n")
for i, d in enumerate(docs, 1):
    print(f"[{i}] {d.page_content}")
    print()


Top results for: 'How has my music taste evolved over time?'

[1] Track: take your vibes and go by Kito (Album: take your vibes and go). Played 27 times, totalling 47.2 minutes. Skipped 6 times (22% skip rate). Most often listened to in the night. Active years: 2024. First played 2024-09-12, last played 2024-11-03. Usually played on windows. Played on shuffle 89% of the time.

[2] Track: feel U luv Me by Knock2 (Album: feel U luv Me). Played 6 times, totalling 15.6 minutes. Skipped 2 times (33% skip rate). Most often listened to in the night. Active years: 2024, 2025. First played 2024-11-17, last played 2025-01-04. Usually played on windows. Played on shuffle 67% of the time.

[3] Track: i don't really feel it anymore by glaive (Album: a bit of a mad one). Played 7 times, totalling 11.1 minutes. Skipped 1 times (14% skip rate). Most often listened to in the night. Active years: 2024. First played 2024-04-02, last played 2024-05-01. Usually played on windows. Played on shuffle 100% of

In [8]:
from langchain_ollama import OllamaLLM

llm = OllamaLLM(model="llama3.2")

query = "How has my music taste evolved over time?"

docs = retriever.invoke(query)

# Combine context
context = "\n\n".join([d.page_content for d in docs])

prompt = f"""
You are analyzing a user's Spotify listening history.

Based on the following data, answer the question:
"{query}"

Focus on trends over time, changes in behavior, and patterns.

DATA:
{context}

Answer:
"""

response = llm.invoke(prompt)

print(response)

Based on the provided data, here's an analysis of how your music taste has evolved over time:

**Initial Assessment:**
Your listening history is dominated by nighttime tracks, suggesting that you might be a night owl or enjoy relaxing during this time. The majority of these tracks are played on Windows devices.

**Emergence of Patterns:**

1. **Shuffle Preferences:** Your shuffle preference varies across tracks. For some songs (like "take your vibes and go" and "i don't really feel it anymore"), you prefer 100% shuffle, indicating a willingness to discover new artists or tracks in an unexpected order. In contrast, other songs ("feel U luv Me" and "i like the way you kiss me") are played on shuffle only about 67-89% of the time.
2. **Device Switching:** You primarily listen to music on Windows devices (4 out of 5 tracks), with the exception of one track played on iOS. This might indicate a strong affinity for the operating system or hardware.
3. **Skip Rates:** Some songs have higher sk

In [13]:
from tools import listening_trends_by_year
from langchain_ollama import OllamaLLM


# Run tool
trend_data = listening_trends_by_year(df)

# Initialize LLM
llm = OllamaLLM(model="llama3.2")

# Create prompt
prompt = f"""
You are analyzing a user's Spotify listening history.

IMPORTANT RULES:
- ONLY use the data provided
- DO NOT make assumptions beyond the data
- DO NOT mention seasonal trends or general user behavior
- If something is not directly supported, say "insufficient data"
- DO NOT calculate percentages unless explicitly provided

DATA:
{trend_data}

Your task:
- Describe how listening time changes year to year
- Identify clear increases or decreases
- Point out anomalies (e.g., sudden spikes or drops)

Be precise and grounded in the data.

Answer:
"""

response = llm.invoke(prompt)

print(response)

Based on the provided Spotify listening history data, here are the observations:

1. The overall trend shows a steady increase in listening time from 2017 to 2023.
2. A clear increase can be observed from 2018 to 2023, with the largest jump occurring between 2020 and 2023.
3. Another notable increase is seen between 2021 and 2023.
4. The listening time increases in each subsequent year until 2023.
5. There's a significant drop from 2023 to 2025.

No other clear increases or decreases can be identified, except for the anomalies mentioned above (sudden spikes or drops).
